# Setup

In [373]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import io
import numpy as np
from fredapi import Fred
import yfinance as yf
import quantreo.features_engineering as fe

In [374]:
load_dotenv(override=True)

if os.getenv('ALPHAVANTAGE_PREMIUM_API_KEY'):
    print("✓ Alpha Vantage Premium API KEY loaded successfully")
    premiumapikey = os.getenv('ALPHAVANTAGE_PREMIUM_API_KEY')

elif os.getenv('ALPHAVANTAGE_API_KEY'):
    print("✓ Alpha Vantage API KEY loaded successfully")
    apikey = os.getenv('ALPHAVANTAGE_API_KEY')

else:
    print("✗ Error: Alpha Vantage API key not found")

if os.getenv('FRED_API_KEY'):
    print("✓ FRED API KEY loaded successfully")
    fredapikey = os.getenv('FRED_API_KEY')
else:
    print("✗ Error: FRED API key not found")


✓ Alpha Vantage Premium API KEY loaded successfully
✓ FRED API KEY loaded successfully


In [375]:
if premiumapikey:
    alphaapikey = premiumapikey
    print("✓ Alpha Vantage Premium API KEY set successfully")
else:
    alphaapikey = apikey
    print("✓ Alpha Vantage API KEY set successfully")

✓ Alpha Vantage Premium API KEY set successfully


# Finance Data Alpha Vantage Setup

In [376]:
baseUrl= "https://www.alphavantage.co/query?"

symbols = ["0QKI.LON","0QLR.LON","NSRGY","RHO6.FRK","ABBNY","UBS","0QP2.LON","0QKY.LON","0QNO.LON","0QPS.LON","0A0D.LON","0Z4C.LON","0QOQ.LON","0QMG.LON","0QQ2.LON","0QMW.LON","0QK6.LON"]
print("Symbols: ",len(symbols))


symbol = "UBS"
interval = "daily" # Maybe not needed anymore
datatype = "csv"


Symbols:  17


# Daily Symbol Data ohlcv

In [377]:
function = "TIME_SERIES_DAILY" #TIME_SERIES_DAILY
adjusted = "true"
extended_hours = "true"
outputsize = "full"

url = (
    f"{baseUrl}"
    f"function={function}&symbol={symbol}"
    f"&outputsize={outputsize}&datatype={datatype}"
    f"&apikey={alphaapikey}"
)

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:300])

df_ohlcv = pd.read_csv(io.StringIO(response.text))
df_ohlcv["timestamp"] = pd.to_datetime(df_ohlcv["timestamp"])
df_ohlcv = df_ohlcv.set_index("timestamp")
print(df_ohlcv.head())
df_ohlcv.head()
print(df_ohlcv.sort_index(ascending=True))

Status Code: 200
Raw CSV Data:
timestamp,open,high,low,close,volume
2025-11-28,38.3100,38.6800,38.3000,38.6000,1039768
2025-11-26,38.0500,38.1950,37.9850,38.0500,1192606
2025-11-25,37.2900,37.6400,37.0950,37.5900,1350342
2025-11-24,36.6100,36.8950,36.4435,36.7900,1973760
2025-11-21,37.0400,37.2000,36.6550,37.0700,1989950
20
             open    high      low  close   volume
timestamp                                         
2025-11-28  38.31  38.680  38.3000  38.60  1039768
2025-11-26  38.05  38.195  37.9850  38.05  1192606
2025-11-25  37.29  37.640  37.0950  37.59  1350342
2025-11-24  36.61  36.895  36.4435  36.79  1973760
2025-11-21  37.04  37.200  36.6550  37.07  1989950
             open    high      low    close   volume
timestamp                                           
2014-11-21  17.47  17.470  17.3900  17.3900     7000
2014-11-24  17.55  17.990  17.3800  17.5800     6832
2014-11-25  17.56  17.560  17.4835  17.4835     4426
2014-11-26  17.55  23.200  17.5200  23.2000     2817

In [378]:
print(url)

https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=UBS&outputsize=full&datatype=csv&apikey=BFMDNSKEEDKULH7S


## Data merge

In [379]:
data = df_ohlcv.sort_index(ascending=True)

In [380]:
# Dataframe Preparation or Reset
df_target = df_ohlcv.sort_index(ascending=True)
df_calc = df_ohlcv.sort_index(ascending=True)
df_macro = df_ohlcv.sort_index(ascending=True)

# Daily Symbols Data Close

In [381]:
df_symbols = pd.DataFrame()

for s in [x for x in symbols if x != symbol]:
    function = "TIME_SERIES_DAILY" #TIME_SERIES_DAILY
    adjusted = "true"
    extended_hours = "true"
    outputsize = "full"

    url = (
        f"{baseUrl}"
        f"function={function}&symbol={s}"
        f"&outputsize={outputsize}&datatype={datatype}"
        f"&apikey={alphaapikey}"
    )

    response = requests.get(url)
    print("Status Code:",response.status_code)
    print("Raw CSV Data:")
    print(response.text[:300])

    df = pd.read_csv(io.StringIO(response.text))
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    print(df.head())
    df.head()
    print(df.sort_index(ascending=True))

    df_symbols[s]= df["close"]

Status Code: 200
Raw CSV Data:
timestamp,open,high,low,close,volume
2025-11-28,581.5000,582.0000,576.5000,582.0000,97
2025-11-27,584.5000,596.5000,579.0000,583.0087,17088
2025-11-26,582.5000,585.5000,579.5000,581.6965,14514
2025-11-25,577.7500,589.2500,576.5000,583.2849,10441
2025-11-24,575.7500,580.6162,571.5000,580.5000,10
              open      high    low     close  volume
timestamp                                            
2025-11-28  581.50  582.0000  576.5  582.0000      97
2025-11-27  584.50  596.5000  579.0  583.0087   17088
2025-11-26  582.50  585.5000  579.5  581.6965   14514
2025-11-25  577.75  589.2500  576.5  583.2849   10441
2025-11-24  575.75  580.6162  571.5  580.5000  105799
                open      high       low     close  volume
timestamp                                                 
2006-03-24  425.9913  430.0105  425.9913  430.0105     210
2006-04-05  423.5180  425.0704  423.5180  425.0704      19
2006-06-02  400.1713  400.6263  400.1713  400.6263       3


## Data merge

In [382]:
data = data.drop(columns=df_symbols.columns.intersection(data.columns))

data = data.merge(df_symbols, left_index=True, right_index=True, how="left")
data.sort_index(ascending=True).head()

,open,high,low,close,volume,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,...,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,0Z4C.LON,0QOQ.LON,0QMG.LON,0QQ2.LON,0QMW.LON,0QK6.LON
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-21,17.47,17.47,17.3900,17.3900,7000,579.2500,77.0812,74.305,30.084,22.76,...,72.4604,NaN,1665.90,NaN,NaN,NaN,NaN,329.39,NaN,NaN
2014-11-24,17.55,17.99,17.3800,17.5800,6832,579.7741,76.8842,74.300,29.888,22.95,...,72.1500,NaN,1654.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-25,17.56,17.56,17.4835,17.4835,4426,580.9120,76.6146,74.250,29.976,23.02,...,72.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0
2014-11-26,17.55,23.20,17.5200,23.2000,2817,580.0000,76.8697,74.809,29.866,22.89,...,71.5130,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-28,18.09,18.13,17.9600,17.9900,111605,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Technical Indicators

### SMA

In [383]:
function= "SMA"
interval = "daily" 
#time_periods= ["6","24","72"]
time_period= "6"
series_type= "close"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df2 = pd.read_csv(io.StringIO(response.text))
df2 = df2.rename(columns={"time": "timestamp"})
df2["timestamp"] = pd.to_datetime(df2["timestamp"])
df2 = df2.set_index("timestamp")
print(df2.head())
print(df2.sort_index(ascending=True).head())


Status Code: 200
Raw CSV Data:
time,SMA
2025-11-28,37.4817
2025-11-26,37.4150
2025-11-25,37.4117
2025-11-24,37.5217
2025-11-21
                SMA
timestamp          
2025-11-28  37.4817
2025-11-26  37.4150
2025-11-25  37.4117
2025-11-24  37.5217
2025-11-21  37.8767
                SMA
timestamp          
2014-12-01  13.0558
2014-12-02  13.1272
2014-12-03  13.1892
2014-12-04  13.2684
2014-12-05  12.6820


### EMA

In [384]:
function= "EMA"
interval = "daily" 
#time_periods= ["6","24"]
time_period= "6"
series_type= "close"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df3 = pd.read_csv(io.StringIO(response.text))
df3 = df3.rename(columns={"time": "timestamp"})
df3["timestamp"] = pd.to_datetime(df3["timestamp"])
df3 = df3.set_index("timestamp")
print(df3.head())
print(df3.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,EMA
2025-11-28,37.9186
2025-11-26,37.6460
2025-11-25,37.4844
2025-11-24,37.4421
2025-11-21
                EMA
timestamp          
2025-11-28  37.9186
2025-11-26  37.6460
2025-11-25  37.4844
2025-11-24  37.4421
2025-11-21  37.7030
                EMA
timestamp          
2014-12-01  13.0558
2014-12-02  12.9373
2014-12-03  12.8747
2014-12-04  12.8400
2014-12-05  12.8213


### RSI

In [385]:
function= "RSI"
interval = "daily" 
#time_periods= ["14"]
time_period= "14"
series_type= "close"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df4 = pd.read_csv(io.StringIO(response.text))
df4 = df4.rename(columns={"time": "timestamp"})
df4["timestamp"] = pd.to_datetime(df4["timestamp"])
df4 = df4.set_index("timestamp")
print(df4.head())
print(df4.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,RSI
2025-11-28,52.2360
2025-11-26,47.8205
2025-11-25,43.7847
2025-11-24,35.7607
2025-11-21
                RSI
timestamp          
2025-11-28  52.2360
2025-11-26  47.8205
2025-11-25  43.7847
2025-11-24  35.7607
2025-11-21  37.5003
                RSI
timestamp          
2014-12-12  50.0401
2014-12-15  49.1066
2014-12-16  49.2918
2014-12-17  50.0263
2014-12-18  51.1937


### ATR

In [386]:
function= "ATR"
interval = "daily" 
#time_periods= ["14"]
time_period= "14"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}"
        f"&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df5 = pd.read_csv(io.StringIO(response.text))
df5 = df5.rename(columns={"time": "timestamp"})
df5["timestamp"] = pd.to_datetime(df5["timestamp"])
df5 = df5.set_index("timestamp")
print(df5.head())
print(df5.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,ATR
2025-11-28,0.6802
2025-11-26,0.6841
2025-11-25,0.6902
2025-11-24,0.6779
2025-11-21,0.6
               ATR
timestamp         
2025-11-28  0.6802
2025-11-26  0.6841
2025-11-25  0.6902
2025-11-24  0.6779
2025-11-21  0.6819
               ATR
timestamp         
2014-12-12  0.6952
2014-12-15  0.6651
2014-12-16  0.6366
2014-12-17  0.6052
2014-12-18  0.5740


### BBANDS

In [387]:
function= "BBANDS"
interval = "daily" 
#time_periods= ["20"]
time_period= "20"
series_type= "close"
nbdevup= "2"
nbdevdn= "2"


url = (f"{baseUrl}"
        f"function={function}&symbol={symbol}&interval={interval}"
        f"&time_period={time_period}&series_type={series_type}"
        f"&datatype={datatype}&apikey={alphaapikey}"
        f"&nbdevup={nbdevup}&nbdevdn={nbdevdn}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df6 = pd.read_csv(io.StringIO(response.text))
df6 = df6.rename(columns={"time": "timestamp"})
df6["timestamp"] = pd.to_datetime(df6["timestamp"])
df6 = df6.set_index("timestamp")
print(df6.head())
print(df6.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
time,Real Lower Band,Real Middle Band,Real Upper Band
2025-11-28,36.7528,38.1645,39.5762
2025-11-2
            Real Lower Band  Real Middle Band  Real Upper Band
timestamp                                                     
2025-11-28          36.7528           38.1645          39.5762
2025-11-26          36.7377           38.1360          39.5343
2025-11-25          36.7484           38.1475          39.5466
2025-11-24          36.7906           38.2225          39.6544
2025-11-21          37.0340           38.3130          39.5920
            Real Lower Band  Real Middle Band  Real Upper Band
timestamp                                                     
2014-12-19          10.8880           12.6271          14.3662
2014-12-22          10.8971           12.6320          14.3670
2014-12-23          10.8922           12.6292          14.3662
2014-12-24          10.8998           12.6336          14.3675
2014-12-26          12.0066           12.4395      

## df merge

In [388]:
dfs = [df2, df3, df4, df5, df6]

df_ti= dfs[0]

for df in dfs[1:]:
    df_ti = df_ti.merge(df, on="timestamp", how="left")

df_ti = df_ti.sort_index(ascending=True)
df_ti.head()

,SMA,EMA,RSI,ATR,Real Lower Band,Real Middle Band,Real Upper Band
timestamp,,,,,,,
2014-12-01,13.0558,13.0558,NaN,NaN,NaN,NaN,NaN
2014-12-02,13.1272,12.9373,NaN,NaN,NaN,NaN,NaN
2014-12-03,13.1892,12.8747,NaN,NaN,NaN,NaN,NaN
2014-12-04,13.2684,12.8400,NaN,NaN,NaN,NaN,NaN
2014-12-05,12.6820,12.8213,NaN,NaN,NaN,NaN,NaN


## Data Checks

In [389]:
#df_finance["timestamp"] = pd.to_datetime(df_finance["timestamp"])
#df_finance = df_finance.set_index("timestamp")
df = df_ti.sort_index(ascending=True)
print(df.isna().sum().sum())

# for d in dfs:
#     print(d.isna().sum().sum())
#     print(len(d))
#     print(len(d.dropna()))

print(df_ti.dropna().sort_index(ascending=True).head())
print(df_ti.dropna().sort_index(ascending=False).head())




60
                SMA      EMA      RSI     ATR  Real Lower Band  \
timestamp                                                        
2014-12-19  12.1904  12.2663  49.9908  0.5496          10.8880   
2014-12-22  12.2056  12.2791  50.8390  0.5183          10.8971   
2014-12-23  12.2431  12.2822  50.6540  0.4883          10.8922   
2014-12-24  12.2887  12.3064  51.3532  0.4620          10.8998   
2014-12-26  12.3238  12.3358  51.7547  0.4395          12.0066   

            Real Middle Band  Real Upper Band  
timestamp                                      
2014-12-19           12.6271          14.3662  
2014-12-22           12.6320          14.3670  
2014-12-23           12.6292          14.3662  
2014-12-24           12.6336          14.3675  
2014-12-26           12.4395          12.8723  
                SMA      EMA      RSI     ATR  Real Lower Band  \
timestamp                                                        
2025-11-28  37.4817  37.9186  52.2360  0.6802          36.7528   


## Data merge

In [390]:
data = data.drop(columns=df_ti.columns.intersection(data.columns))

data = data.merge(df_ti, left_index=True, right_index=True, how="left")
data.sort_index(ascending=True).head()

,open,high,low,close,volume,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,...,0QQ2.LON,0QMW.LON,0QK6.LON,SMA,EMA,RSI,ATR,Real Lower Band,Real Middle Band,Real Upper Band
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-21,17.47,17.47,17.3900,17.3900,7000,579.2500,77.0812,74.305,30.084,22.76,...,329.39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-24,17.55,17.99,17.3800,17.5800,6832,579.7741,76.8842,74.300,29.888,22.95,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-25,17.56,17.56,17.4835,17.4835,4426,580.9120,76.6146,74.250,29.976,23.02,...,NaN,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-26,17.55,23.20,17.5200,23.2000,2817,580.0000,76.8697,74.809,29.866,22.89,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-11-28,18.09,18.13,17.9600,17.9900,111605,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Target

https://docs.quantreo.com/features-engineering/volatility/

## CTC Volatility

In [391]:
df_target["ctc_vol"] = fe.volatility.close_to_close_volatility(df=df_target, close_col="close", window_size=5)

## Parkinson Volatility

In [392]:
df_target["parkinson_vol"] = fe.volatility.parkinson_volatility(df=df_target, high_col="high", low_col="low", window_size=5)

## Target Shift

In [393]:
df_target["ctc_vol_shifted"] = df_target["ctc_vol"].shift(-1)

In [394]:
df_target["parkinson_vol_shifted"] = df_target["parkinson_vol"].shift(-1)

In [395]:
#df_target["target_vol_next_1d"] = df_target["range_t"].shift(-1)

In [396]:
# df_target["target_vol_next"] = df_target["abs_log_return"].shift(-1)
# df_target["target_vol_next"] = df_target["ret_vol_6"].shift(-1) # For trading this is not so good as abs_log_return

In [397]:
# df_target["target_vol_next_3d"] = df_target["range_t"].shift(-1).rolling(3).mean()
# df_target["target_vol_next_5d"] = df_target["range_t"].shift(-1).rolling(5).mean()

## Data merge

In [398]:
data = data.drop(columns=df_target.columns.intersection(data.columns))

data = data.merge(df_target, left_index=True, right_index=True, how="left")
data.sort_index(ascending=True).head()

,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,Real Upper Band,open,high,low,close,volume,ctc_vol,parkinson_vol,ctc_vol_shifted,parkinson_vol_shifted
timestamp,,,,,,,,,,,,,,,,,,,,,
2014-11-21,579.2500,77.0812,74.305,30.084,22.76,297.5000,72.4604,NaN,1665.90,NaN,...,NaN,17.47,17.47,17.3900,17.3900,7000,NaN,NaN,NaN,NaN
2014-11-24,579.7741,76.8842,74.300,29.888,22.95,298.7200,72.1500,NaN,1654.05,NaN,...,NaN,17.55,17.99,17.3800,17.5800,6832,NaN,NaN,NaN,NaN
2014-11-25,580.9120,76.6146,74.250,29.976,23.02,299.4328,72.0000,NaN,NaN,NaN,...,NaN,17.56,17.56,17.4835,17.4835,4426,NaN,NaN,NaN,NaN
2014-11-26,580.0000,76.8697,74.809,29.866,22.89,299.2405,71.5130,NaN,NaN,NaN,...,NaN,17.55,23.20,17.5200,23.2000,2817,NaN,NaN,NaN,NaN
2014-11-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,18.09,18.13,17.9600,17.9900,111605,NaN,NaN,0.190212,0.076048


# Calculations

## Simple Return

In [399]:
df_calc['return'] = df_calc['close'].pct_change() # Maybe only use Log Return and delete this. They are to similar

## Log Return

In [400]:
df_calc["log_return"] = np.log(df_calc["close"] / df_calc["close"].shift(1))

## Absolute Log Return

In [401]:
df_calc["abs_log_return"] = df_calc["log_return"].abs()
df_calc.head()

,open,high,low,close,volume,return,log_return,abs_log_return
timestamp,,,,,,,,
2014-11-21,17.47,17.47,17.3900,17.3900,7000,NaN,NaN,NaN
2014-11-24,17.55,17.99,17.3800,17.5800,6832,0.010926,0.010867,0.010867
2014-11-25,17.56,17.56,17.4835,17.4835,4426,-0.005489,-0.005504,0.005504
2014-11-26,17.55,23.20,17.5200,23.2000,2817,0.326965,0.282895,0.282895
2014-11-28,18.09,18.13,17.9600,17.9900,111605,-0.224569,-0.254336,0.254336


## Current candle range

In [402]:
df_calc["range_t"] = np.log(df_calc["high"] / df_calc["low"])

## Candle Shape

In [403]:
df_calc["body"] = (df_calc["close"] - df_calc["open"]).abs()

df_calc["upper_wick"] = df_calc["high"] - df_calc[["open", "close"]].max(axis=1)
df_calc["lower_wick"] = df_calc[["open", "close"]].min(axis=1) - df_calc["low"]

In [404]:
eps = 1e-9 # Safeguard so we never divide by 0
range_ = df_calc["high"] - df_calc["low"]

df_calc["body_ratio"] = df_calc["body"] / (range_ + eps)
df_calc["upper_wick_ratio"] = df_calc["upper_wick"] / (range_ + eps)
df_calc["lower_wick_ratio"] = df_calc["lower_wick"] / (range_ + eps)
print(1+eps)

1.000000001


## Bollinger Bands Width

In [405]:
eps = 1e-9

df_calc["bb_width"] = df_ti["Real Upper Band"] - df_ti["Real Lower Band"]
df_calc["bb_width_norm"] = df_calc["bb_width"] / (df_ti["Real Middle Band"] + eps)

## Time-based

In [406]:
df_calc["day_of_week"] = df_calc.index.dayofweek

## Data merge

In [407]:
data = data.drop(columns=df_calc.columns.intersection(data.columns))

data = data.merge(df_calc, left_index=True, right_index=True, how="left")
data.sort_index(ascending=False).head()

,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,range_t,body,upper_wick,lower_wick,body_ratio,upper_wick_ratio,lower_wick_ratio,bb_width,bb_width_norm,day_of_week
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-28,582.0000,104.9200,99.58,41.360,71.76,576.0000,74.7700,550.0000,3339.000,63.7600,...,0.009873,0.29,0.080,0.0100,0.763158,0.210526,0.026316,2.8234,0.073980,4
2025-11-26,581.6965,104.8216,98.64,41.685,70.39,574.2851,74.4687,541.0258,3342.334,64.3304,...,0.005513,0.00,0.145,0.0650,0.000000,0.690476,0.309524,2.7966,0.073332,2
2025-11-25,583.2849,104.5367,98.91,41.530,69.13,568.6862,74.5000,538.6921,3322.671,62.8806,...,0.014585,0.30,0.050,0.1950,0.550459,0.091743,0.357798,2.7982,0.073352,1
2025-11-24,580.5000,103.0000,99.51,41.760,68.56,565.2000,72.2708,534.8000,3347.135,61.5800,...,0.012313,0.18,0.105,0.1665,0.398671,0.232558,0.368771,2.8638,0.074924,0
2025-11-21,580.5000,101.8550,100.07,41.900,67.79,558.1000,70.4600,534.2808,3309.500,60.4000,...,0.014759,0.03,0.130,0.3850,0.055046,0.238532,0.706422,2.5580,0.066766,4


In [408]:
# The most current row gets dropped cause of the target value shift 


first_row = data.sort_index(ascending=False).iloc[0]
na_columns = first_row[first_row.isna()].index

print("Date of first row:", data.sort_index(ascending=False).index[0])
print("Columns with NA in first row:", na_columns.tolist())


Date of first row: 2025-11-28 00:00:00
Columns with NA in first row: ['ctc_vol_shifted', 'parkinson_vol_shifted']


# Lagged Features

In [409]:
df_lag = df_calc.sort_index(ascending=True)

## Rolling Volatility

In [410]:
windows = [6,24]

for w in windows:
    df_lag[f'range_mean_{w}'] = df_lag["range_t"].rolling(w).mean()
    df_lag[f'range_std_{w}'] = df_lag["range_t"].rolling(w).std()

## Lagged OHLC(V)

In [411]:
lags = 3

for col in ['open', 'high', 'low', 'close']:
    for lag in range(1, lags + 1):
        df_lag[f'{col}_lag{lag}'] = df_lag[col].shift(lag)

## Lagged Log Return

In [412]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'log_return_lag{lag}'] = df_lag["log_return"].shift(lag)

## Rolling Volatility of Log Returns

In [413]:
windows = [3,6,12,24,72]

for w in windows:
    df_lag[f'ret_vol_{w}'] = df_lag["log_return"].rolling(w).std()

## Realized Variance

In [414]:
df_lag["rv_6"] = (df_lag["log_return"]**2).rolling(6).sum()

## Lagged Absolute Log Return

In [415]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'abs_log_return_lag{lag}'] = df_lag["abs_log_return"].shift(lag)

## Lagged Rolling Volatility of Returns

In [416]:
lags = 3

ret_vol_cols = [c for c in df_lag.columns if c.startswith("ret_vol_")]

for col in ret_vol_cols:
    for lag in range(1, lags + 1):
        df_lag[f'{col}_lag{lag}'] = df_lag[col].shift(lag)

## Lagged Realized Variance

In [417]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'rv_6_lag{lag}'] = df_lag["rv_6"].shift(lag)

## Lagged Candle Range

In [418]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'range_t_lag{lag}'] = df_lag["range_t"].shift(lag)

## Lagged Candle Shape

In [419]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'body_ratio_lag{lag}'] = df_lag["body_ratio"].shift(lag)

## Rolling Volume

In [420]:
eps = 1e-9

df_lag["vol_SMA_6"] = df_lag["volume"].rolling(6).mean()
df_lag["vol_SMA_24"] = df_lag["volume"].rolling(24).mean()
df_lag["rel_volume"] = df_lag["volume"] / (df_lag["vol_SMA_24"] + eps)

## Lagged Volume

In [421]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'rel_volume_lag{lag}'] = df_lag["rel_volume"].shift(lag)

## Lagged SMA

In [422]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'SMA_lag{lag}'] = df_ti["SMA"].shift(lag)

## Lagged EMA

In [423]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'EMA_lag{lag}'] = df_ti["EMA"].shift(lag)

## Lagged RSI

In [424]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'RSI_lag{lag}'] = df_ti["RSI"].shift(lag)

## Lagged ATR

In [425]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'ATR_lag{lag}'] = df_ti["ATR"].shift(lag)

## Lagged Bollinger Bands Width

In [426]:
lags = 3

for lag in range(1, lags + 1):
    df_lag[f'bb_width_norm_lag{lag}'] = df_calc["bb_width_norm"].shift(lag)

## Data merge

In [427]:
data = data.drop(columns=df_lag.columns.intersection(data.columns))

data = data.merge(df_lag, left_index=True, right_index=True, how="left")
data.sort_index(ascending=False).head()

,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,EMA_lag3,RSI_lag1,RSI_lag2,RSI_lag3,ATR_lag1,ATR_lag2,ATR_lag3,bb_width_norm_lag1,bb_width_norm_lag2,bb_width_norm_lag3
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-28,582.0000,104.9200,99.58,41.360,71.76,576.0000,74.7700,550.0000,3339.000,63.7600,...,37.4421,47.8205,43.7847,35.7607,0.6841,0.6902,0.6779,0.073332,0.073352,0.074924
2025-11-26,581.6965,104.8216,98.64,41.685,70.39,574.2851,74.4687,541.0258,3342.334,64.3304,...,37.7030,43.7847,35.7607,37.5003,0.6902,0.6779,0.6819,0.073352,0.074924,0.066766
2025-11-25,583.2849,104.5367,98.91,41.530,69.13,568.6862,74.5000,538.6921,3322.671,62.8806,...,37.9561,35.7607,37.5003,34.5436,0.6779,0.6819,0.6924,0.074924,0.066766,0.060239
2025-11-24,580.5000,103.0000,99.51,41.760,68.56,565.2000,72.2708,534.8000,3347.135,61.5800,...,38.4226,37.5003,34.5436,44.3556,0.6819,0.6924,0.6364,0.066766,0.060239,0.047273
2025-11-21,580.5000,101.8550,100.07,41.900,67.79,558.1000,70.4600,534.2808,3309.500,60.4000,...,38.5116,34.5436,44.3556,42.5280,0.6924,0.6364,0.6581,0.060239,0.047273,0.047755


In [428]:
# The most current row gets dropped cause of the target value shift (target_vol_next_1d)


first_row = data.sort_index(ascending=False).iloc[1]
na_columns = first_row[first_row.isna()].index

print("Date of first row:", data.sort_index(ascending=False).index[0])
print("Columns with NA in first row:", na_columns.tolist())


Date of first row: 2025-11-28 00:00:00
Columns with NA in first row: []


## Drop NaN 

In [429]:
data = data.dropna()
data = data.copy()
print(len(data))
print(data.columns)
data.sort_index(ascending=False).head()

1155
Index(['0QKI.LON', '0QLR.LON', 'NSRGY', 'RHO6.FRK', 'ABBNY', '0QP2.LON',
       '0QKY.LON', '0QNO.LON', '0QPS.LON', '0A0D.LON',
       ...
       'EMA_lag3', 'RSI_lag1', 'RSI_lag2', 'RSI_lag3', 'ATR_lag1', 'ATR_lag2',
       'ATR_lag3', 'bb_width_norm_lag1', 'bb_width_norm_lag2',
       'bb_width_norm_lag3'],
      dtype='object', length=118)


,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,EMA_lag3,RSI_lag1,RSI_lag2,RSI_lag3,ATR_lag1,ATR_lag2,ATR_lag3,bb_width_norm_lag1,bb_width_norm_lag2,bb_width_norm_lag3
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-26,581.6965,104.8216,98.64,41.685,70.39,574.2851,74.4687,541.0258,3342.334,64.3304,...,37.7030,43.7847,35.7607,37.5003,0.6902,0.6779,0.6819,0.073352,0.074924,0.066766
2025-11-25,583.2849,104.5367,98.91,41.530,69.13,568.6862,74.5000,538.6921,3322.671,62.8806,...,37.9561,35.7607,37.5003,34.5436,0.6779,0.6819,0.6924,0.074924,0.066766,0.060239
2025-11-24,580.5000,103.0000,99.51,41.760,68.56,565.2000,72.2708,534.8000,3347.135,61.5800,...,38.4226,37.5003,34.5436,44.3556,0.6819,0.6924,0.6364,0.066766,0.060239,0.047273
2025-11-21,580.5000,101.8550,100.07,41.900,67.79,558.1000,70.4600,534.2808,3309.500,60.4000,...,38.5116,34.5436,44.3556,42.5280,0.6924,0.6364,0.6581,0.060239,0.047273,0.047755
2025-11-20,575.7500,99.9250,98.09,42.140,67.56,562.6000,72.1600,530.0000,3261.500,61.0700,...,38.7043,44.3556,42.5280,44.2755,0.6364,0.6581,0.6687,0.047273,0.047755,0.050977


In [430]:
data.head()

,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,EMA_lag3,RSI_lag1,RSI_lag2,RSI_lag3,ATR_lag1,ATR_lag2,ATR_lag3,bb_width_norm_lag1,bb_width_norm_lag2,bb_width_norm_lag3
timestamp,,,,,,,,,,,,,,,,,,,,,
2019-04-17,468.0,74.8187,94.34,29.300,20.89,327.4,52.3162,292.40,2558.4,57.5217,...,10.0661,70.3688,66.1057,64.4432,0.1523,0.1499,0.1534,0.112705,0.100527,0.092797
2019-04-18,471.2,73.8520,94.71,28.535,20.96,328.2,53.4181,297.67,2565.0,55.9229,...,10.1313,74.0583,70.3688,66.1057,0.1539,0.1523,0.1499,0.129590,0.112705,0.100527
2019-06-20,494.7,86.3439,103.83,31.235,19.79,339.3,49.1238,333.20,2801.0,57.0109,...,9.7797,48.5804,45.1811,37.0193,0.1581,0.1547,0.1505,0.049515,0.055334,0.058886
2019-06-26,488.1,84.3724,102.69,30.585,19.78,340.1,47.7741,323.80,2793.0,57.9634,...,9.8202,42.3686,45.1905,44.3297,0.1413,0.1450,0.1517,0.044355,0.043759,0.046958
2019-06-27,488.3,84.5525,102.76,30.370,19.96,339.4,47.7711,324.60,2722.0,58.6230,...,9.8145,46.8375,42.3686,45.1905,0.1390,0.1413,0.1450,0.044150,0.044355,0.043759


# Macroeconomic Data

## Alpha Vantage API Calls

### Federal Funds

In [431]:
function= "FEDERAL_FUNDS_RATE"

interval = "daily"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df1_macro = pd.read_csv(io.StringIO(response.text))
df1_macro = df1_macro.rename(columns={"time": "timestamp"})
df1_macro = df1_macro.rename(columns={"value": function})
df1_macro["timestamp"] = pd.to_datetime(df1_macro["timestamp"])
df1_macro = df1_macro.set_index("timestamp")
print(df1_macro.head())
print(df1_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-11-27,3.88
2025-11-26,3.88
2025-11-25,3.88
2025-11-24,3.88
2025-11-23,3.88
            FEDERAL_FUNDS_RATE
timestamp                     
2025-11-27                3.88
2025-11-26                3.88
2025-11-25                3.88
2025-11-24                3.88
2025-11-23                3.88
            FEDERAL_FUNDS_RATE
timestamp                     
1954-07-01                1.13
1954-07-02                1.25
1954-07-03                1.25
1954-07-04                1.25
1954-07-05                0.88


### Inflation

In [432]:
function= "INFLATION"

url = (f"{baseUrl}"
        f"function={function}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df2_macro = pd.read_csv(io.StringIO(response.text))
df2_macro = df2_macro.rename(columns={"value": function})
df2_macro["timestamp"] = pd.to_datetime(df2_macro["timestamp"])
df2_macro = df2_macro.set_index("timestamp")
print(df2_macro.head())
print(df2_macro.sort_index(ascending=True).head())

# test = pd.DataFrame(index=df_finance.index)
# test = test.merge(df2_macro, on="timestamp", how="left")
# # test = test.ffill()
# test = test.interpolate(method="time")
# test.head(479)

Status Code: 200
Raw CSV Data:
timestamp,value
2024-01-01,2.94952520485207
2023-01-01,4.11633838374488
2022-01-01,8.002799820521
            INFLATION
timestamp            
2024-01-01   2.949525
2023-01-01   4.116338
2022-01-01   8.002800
2021-01-01   4.697859
2020-01-01   1.233584
            INFLATION
timestamp            
1960-01-01   1.457976
1961-01-01   1.070724
1962-01-01   1.198773
1963-01-01   1.239669
1964-01-01   1.278912


### Real GDP

In [433]:
function= "REAL_GDP"

interval = "quarterly"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df3_macro = pd.read_csv(io.StringIO(response.text))
df3_macro = df3_macro.rename(columns={"time": "timestamp"})
df3_macro = df3_macro.rename(columns={"value": function})
df3_macro["timestamp"] = pd.to_datetime(df3_macro["timestamp"])
df3_macro = df3_macro.set_index("timestamp")
print(df3_macro.head())
print(df3_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-04-01,5943.384
2025-01-01,5776.724
2024-10-01,5997.184
2024-07-01,5884.567
            REAL_GDP
timestamp           
2025-04-01  5943.384
2025-01-01  5776.724
2024-10-01  5997.184
2024-07-01  5884.567
2024-04-01  5829.384
            REAL_GDP
timestamp           
2002-01-01  3501.118
2002-04-01  3608.496
2002-07-01  3650.253
2002-10-01  3712.845
2003-01-01  3582.767


### Real GDP per Capita

In [434]:
function= "REAL_GDP_PER_CAPITA"

url = (f"{baseUrl}"
        f"function={function}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df4_macro = pd.read_csv(io.StringIO(response.text))
df4_macro = df4_macro.rename(columns={"time": "timestamp"})
df4_macro = df4_macro.rename(columns={"value": function})
df4_macro["timestamp"] = pd.to_datetime(df4_macro["timestamp"])
df4_macro = df4_macro.set_index("timestamp")
print(df4_macro.head())
print(df4_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-04-01,69499.0
2025-01-01,68937.0
2024-10-01,69136.0
2024-07-01,68926.0
202
            REAL_GDP_PER_CAPITA
timestamp                      
2025-04-01              69499.0
2025-01-01              68937.0
2024-10-01              69136.0
2024-07-01              68926.0
2024-04-01              68504.0
            REAL_GDP_PER_CAPITA
timestamp                      
1947-01-01              15248.0
1947-04-01              15139.0
1947-07-01              15039.0
1947-10-01              15204.0
1948-01-01              15371.0


### Treasury Yield

In [435]:
function= "TREASURY_YIELD"

interval = "daily"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df5_macro = pd.read_csv(io.StringIO(response.text))
df5_macro = df5_macro.rename(columns={"time": "timestamp"})
df5_macro = df5_macro.rename(columns={"value": function})
df5_macro["timestamp"] = pd.to_datetime(df5_macro["timestamp"])
df5_macro = df5_macro.set_index("timestamp")
print(df5_macro.head())
print(df5_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-11-26,4.0
2025-11-25,4.01
2025-11-24,4.04
2025-11-21,4.06
2025-11-20,4.1

           TREASURY_YIELD
timestamp                
2025-11-26            4.0
2025-11-25           4.01
2025-11-24           4.04
2025-11-21           4.06
2025-11-20            4.1
           TREASURY_YIELD
timestamp                
1962-01-02           4.06
1962-01-03           4.03
1962-01-04           3.99
1962-01-05           4.02
1962-01-08           4.03


### CPI

In [436]:
function= "CPI"

interval = "monthly"

url = (f"{baseUrl}"
        f"function={function}&interval={interval}&datatype={datatype}&apikey={alphaapikey}")

response = requests.get(url)
print("Status Code:",response.status_code)
print("Raw CSV Data:")
print(response.text[:100])

df6_macro = pd.read_csv(io.StringIO(response.text))
df6_macro = df6_macro.rename(columns={"time": "timestamp"})
df6_macro = df6_macro.rename(columns={"value": function})
df6_macro["timestamp"] = pd.to_datetime(df6_macro["timestamp"])
df6_macro = df6_macro.set_index("timestamp")
print(df6_macro.head())
print(df6_macro.sort_index(ascending=True).head())

Status Code: 200
Raw CSV Data:
timestamp,value
2025-09-01,324.800
2025-08-01,323.976
2025-07-01,323.048
2025-06-01,322.561
202
                CPI
timestamp          
2025-09-01  324.800
2025-08-01  323.976
2025-07-01  323.048
2025-06-01  322.561
2025-05-01  321.465
            CPI
timestamp      
1913-01-01  9.8
1913-02-01  9.8
1913-03-01  9.8
1913-04-01  9.8
1913-05-01  9.7


## yfinance API Calls

### yfinance

In [437]:
indices = ["^FTSE","^STOXX50E","^GSPC"]


df7_macro = yf.download(
    tickers=indices,
    period="max",
    interval="1d",
    auto_adjust=True
)["Close"]

# Rename to macro-style column names
df7_macro.rename(columns={
    "^FTSE": "FTSE_100",
    "^STOXX50E": "EUROSTOXX_50",
    "^GSPC": "SP500"
}, inplace=True)

print(df7_macro.head())
df7_macro.sort_index(ascending=False).head()


df7_macro.index = pd.to_datetime(df7_macro.index)
df7_macro.index.name = "timestamp"

print(df7_macro.sort_index(ascending=False).head())
print(df7_macro.sort_index(ascending=True).head())


[*********************100%***********************]  3 of 3 completed

Ticker      FTSE_100      SP500  EUROSTOXX_50
Date                                         
1927-12-30       NaN  17.660000           NaN
1928-01-03       NaN  17.760000           NaN
1928-01-04       NaN  17.719999           NaN
1928-01-05       NaN  17.549999           NaN
1928-01-06       NaN  17.660000           NaN
Ticker         FTSE_100        SP500  EUROSTOXX_50
timestamp                                         
2025-11-28  9720.500000  6849.089844   5668.169922
2025-11-27  9693.900391          NaN   5653.169922
2025-11-26  9691.599609  6812.609863   5655.580078
2025-11-25  9609.500000  6765.879883   5573.910156
2025-11-24  9534.900391  6705.120117   5528.669922
Ticker      FTSE_100      SP500  EUROSTOXX_50
timestamp                                    
1927-12-30       NaN  17.660000           NaN
1928-01-03       NaN  17.760000           NaN
1928-01-04       NaN  17.719999           NaN
1928-01-05       NaN  17.549999           NaN
1928-01-06       NaN  17.660000           NaN

## FRED API Calls

### Fred API Setup

In [438]:
fred = Fred(api_key=fredapikey)
df_macro = df_ohlcv

### Fred API Combined API Call

In [441]:
series_map = {
    "VIX": "VIXCLS",
    #"S&P500": "SP500",
    "CHFUSD": "DEXSZUS",
    "EURUSD": "DEXUSEU",
    "US1Y": "DGS1",
    "US2Y": "DGS2",
    "US5Y": "DGS5",
    "US10Y": "DGS10",
    "US30Y": "DGS30", 
    "YC_Slope": "T10Y2Y",
    "Stress Index": "STLFSI4",
    "FEDFUNDS": "FEDFUNDS",
    "UNRATE": "UNRATE",
    "Median CPI": "MEDCPIM158SFRBCLE",
    "UMCSENT": "UMCSENT",
    "USEPUINDXD": "USEPUINDXD",
}

macro_fred = pd.DataFrame()

for name, sid in series_map.items():
    s = fred.get_series(sid)
    s.index = pd.to_datetime(s.index)
    macro_fred[name] = s

macro_fred.index.name = "timestamp"
macro_fred = macro_fred.sort_index()

df8_macro = macro_fred


# # 1) Resample to daily
# macro_daily = macro_fred.resample("D").ffill()  # carry last known value forward

# # 2) Shift by 1 day so you only use info that was available at t-1
# macro_avail = pd.DataFrame(index=macro_daily.index)

# for name in series_map.keys():
#     macro_avail[f"{name}_avail"] = macro_daily[name].shift(1)

# # shift for information availability (no look-ahead)
# for name in series_map.keys():
#     macro_fred = macro_fred.interpolate(method="time")
#     macro_fred[f"{name}_avail"] = macro_fred[name].shift(1)

# # resample to your frequency (e.g. 60min or daily)
# macro_1d = macro_fred[[f"{name}_avail" for name in series_map.keys()]]
# # macro_1d = macro_1d.interpolate(method="time") # .resample("D").ffill()  

# # merge with your main data
# # data = data.merge(macro_1d, left_index=True, right_index=True, how="left")
# macro_1d.head()
# macro_1d.sort_index(ascending=True).head()


## DF merge

In [442]:
dfs = [df1_macro, df2_macro, df3_macro, df4_macro, df5_macro, df6_macro, df7_macro, df8_macro]

#df_macro = pd.DataFrame(index=df_finance.index)
df_macro = dfs[0]

for df in dfs[1:]:
#for df in dfs:
    df_macro = df_macro.merge(df, on="timestamp", how="outer")

df_macro.sort_index(ascending=False).head(365)

,FEDERAL_FUNDS_RATE,INFLATION,REAL_GDP,REAL_GDP_PER_CAPITA,TREASURY_YIELD,CPI,FTSE_100,SP500,EUROSTOXX_50,VIX,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-28,NaN,NaN,NaN,NaN,NaN,NaN,9720.500000,6849.089844,5668.169922,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-11-27,3.88,NaN,NaN,NaN,NaN,NaN,9693.900391,NaN,5653.169922,17.21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,245.45
2025-11-26,3.88,NaN,NaN,NaN,4.0,NaN,9691.599609,6812.609863,5655.580078,17.19,...,3.56,4.00,4.64,0.55,NaN,NaN,NaN,NaN,NaN,243.59
2025-11-25,3.88,NaN,NaN,NaN,4.01,NaN,9609.500000,6765.879883,5573.910156,18.56,...,3.55,4.01,4.67,0.58,NaN,NaN,NaN,NaN,NaN,322.00
2025-11-24,3.88,NaN,NaN,NaN,4.04,NaN,9534.900391,6705.120117,5528.669922,20.52,...,3.61,4.04,4.68,0.58,NaN,NaN,NaN,NaN,NaN,189.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-03,4.58,NaN,NaN,NaN,4.23,NaN,8359.400391,6049.879883,4878.509766,13.30,...,4.11,4.23,4.40,0.06,NaN,NaN,NaN,NaN,NaN,92.19
2024-12-02,4.58,NaN,NaN,NaN,4.19,NaN,8312.900391,6047.149902,4846.729980,13.34,...,4.08,4.19,4.36,0.02,NaN,NaN,NaN,NaN,NaN,78.68
2024-12-01,4.58,NaN,NaN,NaN,NaN,315.605,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [443]:
df_macro.columns

Index(['FEDERAL_FUNDS_RATE', 'INFLATION', 'REAL_GDP', 'REAL_GDP_PER_CAPITA',
       'TREASURY_YIELD', 'CPI', 'FTSE_100', 'SP500', 'EUROSTOXX_50', 'VIX',
       'CHFUSD', 'EURUSD', 'US1Y', 'US2Y', 'US5Y', 'US10Y', 'US30Y',
       'YC_Slope', 'Stress Index', 'FEDFUNDS', 'UNRATE', 'Median CPI',
       'UMCSENT', 'USEPUINDXD'],
      dtype='object')

## Publication Lag

https://www.bls.gov/schedule/2025/11_sched.htm  
https://www.bea.gov/news/schedule/full  
https://data.sca.isr.umich.edu/release-schedule  
https://www.policyuncertainty.com/index.html  
https://www.newyorkfed.org/markets/reference-rates/effr  

In [444]:
# Ensure datetime index and sorted
df_macro.index = pd.to_datetime(df_macro.index)
df_macro = df_macro.sort_index()

# Convert all columns to numeric safely
df_macro = df_macro.apply(pd.to_numeric, errors="coerce")

pub_lags = {
    "Federal Funds Rate": 1,
    "INFLATION": 14, 
    "REAL_GDP": 30,
    "REAL_GDP_PER_CAPITA": 30,
    "TREASURY_YIELD": 0,
    "CPI": 14,
    "FTSE_100": 0,
    "EUROSTOXX_50": 0,
    "SP500": 0,
    "VIX": 0,
    "CHFUSD": 1,
    "EURUSD": 1,
    "US1Y": 1,
    "US2Y": 1,
    "US5Y": 1,
    "US10Y": 1,
    "US30Y": 1,
    "YC_Slope": 0,
    "Stress Index": 6,
    "FEDFUNDS": 1,
    "UNRATE": 7,
    "Median CPI": 14,
    "UMCSENT": 15,
    "USEPUINDXD": 30,
}

for col in df_macro.columns:
    if col in pub_lags:
        df_macro[col] = df_macro[col].shift(pub_lags[col])

## Imputation

In [445]:
# Reindex to daily grid
full_index = pd.date_range(
    start=df_macro.index.min(),
    end=df_macro.index.max(),
    freq="D"
)

df_macro = df_macro.reindex(full_index)
df_macro.index.name = "timestamp"

# Safe imputation (no future leakage)
df_macro = df_macro.ffill()
#df_macro = df_macro.interpolate(method="time")
df_macro.sort_index(ascending=False).head(365)

,FEDERAL_FUNDS_RATE,INFLATION,REAL_GDP,REAL_GDP_PER_CAPITA,TREASURY_YIELD,CPI,FTSE_100,SP500,EUROSTOXX_50,VIX,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-28,3.88,2.949525,5943.384,69499.0,4.00,324.800,9720.500000,6849.089844,5668.169922,17.21,...,3.56,4.00,4.64,0.55,-0.1331,4.09,4.4,2.384737,53.6,260.35
2025-11-27,3.88,2.949525,5943.384,69499.0,4.00,324.800,9693.900391,6812.609863,5653.169922,17.21,...,3.56,4.00,4.64,0.55,-0.1331,4.09,4.4,2.384737,53.6,269.14
2025-11-26,3.88,2.949525,5943.384,69499.0,4.00,324.800,9691.599609,6812.609863,5655.580078,17.19,...,3.55,4.01,4.67,0.55,-0.5071,4.09,4.4,2.384737,53.6,386.20
2025-11-25,3.88,2.949525,5943.384,69499.0,4.01,324.800,9609.500000,6765.879883,5573.910156,18.56,...,3.61,4.04,4.68,0.58,-0.5071,4.09,4.4,2.384737,53.6,327.22
2025-11-24,3.88,2.949525,5943.384,69499.0,4.04,324.800,9534.900391,6705.120117,5528.669922,20.52,...,3.62,4.06,4.71,0.58,-0.5071,4.09,4.4,2.384737,53.6,327.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-03,4.58,2.949525,5997.184,69136.0,4.23,315.493,8359.400391,6049.879883,4878.509766,13.30,...,4.08,4.19,4.36,0.06,-0.6114,4.64,4.2,3.226070,71.8,116.29
2024-12-02,4.58,2.949525,5997.184,69136.0,4.19,315.493,8312.900391,6047.149902,4846.729980,13.34,...,4.05,4.18,4.36,0.02,-0.6114,4.64,4.2,3.226070,71.8,116.29
2024-12-01,4.58,2.949525,5997.184,69136.0,4.18,315.493,8287.299805,6032.379883,4804.399902,13.51,...,4.05,4.18,4.36,0.05,-0.6114,4.64,4.2,3.226070,71.8,116.29


In [446]:
df_macro.isna().sum()

FEDERAL_FUNDS_RATE     15156
INFLATION              17180
REAL_GDP               32537
REAL_GDP_PER_CAPITA    12459
TREASURY_YIELD         17898
CPI                      424
FTSE_100               25934
SP500                   5476
EUROSTOXX_50           34421
VIX                    28125
CHFUSD                 28126
EURUSD                 31415
US1Y                   28126
US2Y                   28126
US5Y                   28126
US10Y                  28126
US30Y                  28126
YC_Slope               28125
Stress Index           29590
FEDFUNDS               28156
UNRATE                 28162
Median CPI             28169
UMCSENT                28170
USEPUINDXD             28155
dtype: int64

In [447]:
df_macro.loc[:, df_macro.isna().any(axis=0)].head(132)

,FEDERAL_FUNDS_RATE,INFLATION,REAL_GDP,REAL_GDP_PER_CAPITA,TREASURY_YIELD,CPI,FTSE_100,SP500,EUROSTOXX_50,VIX,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
1913-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1913-05-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-05-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-05-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data merge

In [448]:
data = data.drop(columns=df_macro.columns.intersection(data.columns))

data = data.merge(df_macro, left_index=True, right_index=True, how="left")
data.sort_index(ascending=False).head()

,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-26,581.6965,104.8216,98.64,41.685,70.39,574.2851,74.4687,541.0258,3342.334,64.3304,...,3.55,4.01,4.67,0.55,-0.5071,4.09,4.4,2.384737,53.6,386.20
2025-11-25,583.2849,104.5367,98.91,41.530,69.13,568.6862,74.5000,538.6921,3322.671,62.8806,...,3.61,4.04,4.68,0.58,-0.5071,4.09,4.4,2.384737,53.6,327.22
2025-11-24,580.5000,103.0000,99.51,41.760,68.56,565.2000,72.2708,534.8000,3347.135,61.5800,...,3.62,4.06,4.71,0.58,-0.5071,4.09,4.4,2.384737,53.6,327.22
2025-11-21,580.5000,101.8550,100.07,41.900,67.79,558.1000,70.4600,534.2808,3309.500,60.4000,...,3.68,4.10,4.73,0.55,-0.5071,4.09,4.4,2.384737,53.6,490.22
2025-11-20,575.7500,99.9250,98.09,42.140,67.56,562.6000,72.1600,530.0000,3261.500,61.0700,...,3.71,4.13,4.75,0.55,-0.5071,4.09,4.4,2.384737,53.6,279.64


# Drop Na

In [449]:
data = data.dropna()
data = data.copy()
print(len(data))
print(data.columns)
data.head()

1155
Index(['0QKI.LON', '0QLR.LON', 'NSRGY', 'RHO6.FRK', 'ABBNY', '0QP2.LON',
       '0QKY.LON', '0QNO.LON', '0QPS.LON', '0A0D.LON',
       ...
       'US5Y', 'US10Y', 'US30Y', 'YC_Slope', 'Stress Index', 'FEDFUNDS',
       'UNRATE', 'Median CPI', 'UMCSENT', 'USEPUINDXD'],
      dtype='object', length=142)


,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2019-04-17,468.0,74.8187,94.34,29.300,20.89,327.4,52.3162,292.40,2558.4,57.5217,...,2.41,2.60,2.99,0.19,-0.6766,2.42,3.7,2.829462,97.2,77.23
2019-04-18,471.2,73.8520,94.71,28.535,20.96,328.2,53.4181,297.67,2565.0,55.9229,...,2.40,2.59,2.99,0.19,-0.6600,2.42,3.7,2.829462,97.2,82.83
2019-06-20,494.7,86.3439,103.83,31.235,19.79,339.3,49.1238,333.20,2801.0,57.0109,...,1.77,2.03,2.54,0.29,-0.4377,2.39,3.6,2.558535,100.0,73.80
2019-06-26,488.1,84.3724,102.69,30.585,19.78,340.1,47.7741,323.80,2793.0,57.9634,...,1.73,2.00,2.53,0.28,-0.4377,2.39,3.6,2.558535,100.0,102.04
2019-06-27,488.3,84.5525,102.76,30.370,19.96,339.4,47.7711,324.60,2722.0,58.6230,...,1.80,2.05,2.57,0.27,-0.3504,2.39,3.6,2.558535,100.0,69.48


In [450]:
data.sort_index(ascending=False).head()

,0QKI.LON,0QLR.LON,NSRGY,RHO6.FRK,ABBNY,0QP2.LON,0QKY.LON,0QNO.LON,0QPS.LON,0A0D.LON,...,US5Y,US10Y,US30Y,YC_Slope,Stress Index,FEDFUNDS,UNRATE,Median CPI,UMCSENT,USEPUINDXD
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-11-26,581.6965,104.8216,98.64,41.685,70.39,574.2851,74.4687,541.0258,3342.334,64.3304,...,3.55,4.01,4.67,0.55,-0.5071,4.09,4.4,2.384737,53.6,386.20
2025-11-25,583.2849,104.5367,98.91,41.530,69.13,568.6862,74.5000,538.6921,3322.671,62.8806,...,3.61,4.04,4.68,0.58,-0.5071,4.09,4.4,2.384737,53.6,327.22
2025-11-24,580.5000,103.0000,99.51,41.760,68.56,565.2000,72.2708,534.8000,3347.135,61.5800,...,3.62,4.06,4.71,0.58,-0.5071,4.09,4.4,2.384737,53.6,327.22
2025-11-21,580.5000,101.8550,100.07,41.900,67.79,558.1000,70.4600,534.2808,3309.500,60.4000,...,3.68,4.10,4.73,0.55,-0.5071,4.09,4.4,2.384737,53.6,490.22
2025-11-20,575.7500,99.9250,98.09,42.140,67.56,562.6000,72.1600,530.0000,3261.500,61.0700,...,3.71,4.13,4.75,0.55,-0.5071,4.09,4.4,2.384737,53.6,279.64


# Write to .csv

In [451]:
data.to_csv(f"data_{symbol}.csv", index=True)